In [25]:

import tkinter as tk
from tkinter import ttk, messagebox
import joblib
import numpy as np

# ---------- Load model ----------
try:
    model = joblib.load("movie_recommendation_model.pkl")
    data = model["data"].reset_index(drop=True)
    sim = model["cosine_sim"]

    if sim.shape != (len(data), len(data)):
        raise ValueError("Dataset and similarity matrix dimensions do not match.")

except Exception as e:
    r = tk.Tk()
    r.withdraw()
    messagebox.showerror("Model Error", f"Unable to load model:\n{e}")
    raise SystemExit


titles = data["original_title"].fillna("").astype(str).to_numpy()

index_of = {}

for i, t in enumerate(titles):
    if t:
        index_of.setdefault(t, i)

all_titles = sorted(index_of)

YEAR = "release_date" if "release_date" in data else None
RATING = "vote_average" if "vote_average" in data else None
PLOT = "overview" if "overview" in data else None


# ---------- Movie metadata ----------
def meta(i):
    parts = []

    if YEAR:
        y = str(data.at[i, YEAR])[:4]
        if y.isdigit():
            parts.append(y)

    if RATING:
        try:
            parts.append(f"{float(data.at[i, RATING]):.1f}/10")
        except (TypeError, ValueError):
            pass

    return "  ·  ".join(parts)


def plot(i):
    if not PLOT:
        return ""

    p = str(data.at[i, PLOT])

    return "" if p.lower() == "nan" else p


# ---------- Recommendation function ----------
def recommend(title, n=10):

    i = index_of[title]

    s = np.asarray(sim[i]).ravel().astype(float)

    s[i] = -1

    n = min(n, len(s) - 1)

    top = np.argpartition(s, -n)[-n:]

    top = top[np.argsort(s[top])[::-1]]

    return [(int(j), float(s[j])) for j in top]


# ---------- MATRIX BRAND COLOUR THEME ----------

PAPER = "#0B1D38"
SIDE = "#07152A"
CARD = "#142B49"
LINE = "#263F5F"

INK = "#F5F7FA"
MUTED = "#A9B7CA"

ACCENT = "#FF8A00"
HOVER = "#1C3859"

SERIF = "Georgia"
SANS = "Arial"



# ---------- Main window ----------
root = tk.Tk()

root.title("Matrix Cine - Movie Recommendation System")
root.geometry("1000x640")
root.minsize(760, 500)
root.configure(bg=PAPER)


# ---------- Sidebar ----------
side = tk.Frame(
    root,
    bg=SIDE,
    width=270,
    padx=20,
    pady=24
)

side.pack(side="left", fill="y")
side.pack_propagate(False)


# ==========================================
# MATRIX CINE BRANDING
# ==========================================

tk.Label(
    side,
    text="MATRIX CINE",
    font=("Arial", 22, "bold"),
    fg=ACCENT,
    bg=SIDE,
    anchor="w"
).pack(fill="x")


tk.Label(
    side,
    text="Discover. Match. Watch.",
    font=(SERIF, 11, "italic"),
    fg=INK,
    bg=SIDE,
    anchor="w"
).pack(fill="x", pady=(5, 4))


tk.Label(
    side,
    text="INTELLIGENT MOVIE\nRECOMMENDATION SYSTEM",
    font=("Arial", 7, "bold"),
    fg=MUTED,
    bg=SIDE,
    anchor="w",
    justify="left"
).pack(fill="x", pady=(0, 25))


# ---------- Movie selection heading ----------
tk.Label(
    side,
    text="FILM YOU LIKED",
    font=(SANS, 8, "bold"),
    fg=MUTED,
    bg=SIDE,
    anchor="w"
).pack(fill="x", pady=(0, 6))


# ---------- Styling ----------
style = ttk.Style()

style.theme_use("clam")

style.configure(
    "TCombobox",
    fieldbackground=CARD,
    background=CARD,
    foreground=INK,
    arrowcolor=INK,
    bordercolor=LINE,
    lightcolor=CARD,
    darkcolor=CARD,
    padding=7
)

style.map(
    "TCombobox",
    fieldbackground=[("readonly", CARD)],
    foreground=[("readonly", INK)]
)

style.configure(
    "Vertical.TScrollbar",
    background=LINE,
    troughcolor=PAPER,
    bordercolor=PAPER,
    arrowcolor=MUTED
)

for k, v in (
    ("background", CARD),
    ("foreground", INK),
    ("selectBackground", ACCENT),
    ("selectForeground", "#FFFFFF")
):
    root.option_add(f"*TCombobox*Listbox.{k}", v)


# ---------- Movie selection ----------
var = tk.StringVar()

combo = ttk.Combobox(
    side,
    textvariable=var,
    values=all_titles,
    font=(SANS, 10)
)

combo.pack(fill="x", ipady=3)


# ---------- Buttons ----------
btns = tk.Frame(side, bg=SIDE)

btns.pack(fill="x", pady=12)


def button(text, cmd, primary):

    b = tk.Button(
        btns,
        text=text,
        command=cmd,
        font=(SANS, 9, "bold"),

        fg="#07152A" if primary else INK,
        bg=ACCENT if primary else "#263F5F",
        
        activebackground="#E87900" if primary else "#365477",
        activeforeground="#07152A" if primary else INK,

        bd=0,
        relief="flat",
        padx=14,
        pady=8,
        cursor="hand2"
    )

    b.pack(
        side="left",
        padx=(0, 6),
        fill="x",
        expand=primary
    )


button("Find similar", lambda: run(), True)
button("Clear", lambda: reset(), False)


# ---------- Recently explored ----------
tk.Label(
    side,
    text="RECENTLY EXPLORED",
    font=(SANS, 8, "bold"),
    fg=MUTED,
    bg=SIDE,
    anchor="w"
).pack(fill="x", pady=(18, 6))


history = tk.Listbox(
    side,
    bg=SIDE,
    fg=INK,
    font=(SANS, 10),
    bd=0,
    highlightthickness=0,
    activestyle="none",
    selectbackground="#285052",
    selectforeground=INK,
    cursor="hand2"
)

history.pack(fill="both", expand=True)


# ---------- Main column ----------
main = tk.Frame(root, bg=PAPER)

main.pack(
    side="left",
    fill="both",
    expand=True
)


# ---------- Header ----------
hero = tk.Frame(
    main,
    bg=PAPER,
    padx=36,
    pady=26
)

hero.pack(fill="x")


kicker = tk.Label(
    hero,
    font=(SANS, 8, "bold"),
    fg=ACCENT,
    bg=PAPER,
    anchor="w"
)

kicker.pack(fill="x")


h_title = tk.Label(
    hero,
    font=(SERIF, 26, "bold"),
    fg=INK,
    bg=PAPER,
    anchor="w",
    justify="left"
)

h_title.pack(fill="x", pady=(4, 2))


h_meta = tk.Label(
    hero,
    font=(SANS, 10),
    fg=MUTED,
    bg=PAPER,
    anchor="w"
)

h_meta.pack(fill="x")


h_plot = tk.Label(
    hero,
    font=(SERIF, 10, "italic"),
    fg=INK,
    bg=PAPER,
    anchor="w",
    justify="left"
)

h_plot.pack(fill="x", pady=(8, 0))


def rewrap(e):

    for w in (h_title, h_plot):
        w.config(wraplength=max(200, e.width - 80))


hero.bind("<Configure>", rewrap)


tk.Frame(
    main,
    bg=LINE,
    height=1
).pack(fill="x", padx=36)


# ---------- Results area ----------
wrap = tk.Frame(main, bg=PAPER)

wrap.pack(
    fill="both",
    expand=True,
    padx=(36, 24),
    pady=12
)


canvas = tk.Canvas(
    wrap,
    bg=PAPER,
    highlightthickness=0
)

sb = ttk.Scrollbar(
    wrap,
    orient="vertical",
    command=canvas.yview
)

inner = tk.Frame(
    canvas,
    bg=PAPER
)

win = canvas.create_window(
    (0, 0),
    window=inner,
    anchor="nw"
)

canvas.configure(yscrollcommand=sb.set)

canvas.pack(
    side="left",
    fill="both",
    expand=True
)

sb.pack(
    side="right",
    fill="y"
)


inner.bind(
    "<Configure>",
    lambda e: canvas.configure(
        scrollregion=canvas.bbox("all")
    )
)

canvas.bind(
    "<Configure>",
    lambda e: canvas.itemconfigure(
        win,
        width=e.width
    )
)


root.bind_all(
    "<MouseWheel>",
    lambda e: canvas.yview_scroll(
        int(-e.delta / 120),
        "units"
    )
)

root.bind_all(
    "<Button-4>",
    lambda e: canvas.yview_scroll(-1, "units")
)

root.bind_all(
    "<Button-5>",
    lambda e: canvas.yview_scroll(1, "units")
)


# ---------- Recommendation cards ----------
def make_row(rank, idx, score):

    title = titles[idx]

    row = tk.Frame(
        inner,
        bg=CARD,
        padx=16,
        pady=12,
        cursor="hand2",
        highlightthickness=1,
        highlightbackground=LINE
    )

    row.pack(fill="x", pady=4)


    tk.Label(
        row,
        text=f"{rank}",
        font=(SERIF, 22, "italic"),
        fg=ACCENT,
        bg=CARD,
        width=3
    ).pack(side="left")


    info = tk.Frame(row, bg=CARD)

    info.pack(
        side="left",
        fill="x",
        expand=True,
        padx=(4, 12)
    )


    tk.Label(
        info,
        text=title,
        font=(SERIF, 13, "bold"),
        fg=INK,
        bg=CARD,
        anchor="w"
    ).pack(fill="x")


    tk.Label(
        info,
        text=meta(idx) or "Content-based match",
        font=(SANS, 9),
        fg=MUTED,
        bg=CARD,
        anchor="w"
    ).pack(fill="x")


    right = tk.Frame(row, bg=CARD)

    right.pack(side="right")


    tk.Label(
        right,
        text=f"{score:.0%} alike",
        font=(SANS, 10, "bold"),
        fg=INK,
        bg=CARD
    ).pack(anchor="e")


    meter = tk.Canvas(
        right,
        width=130,
        height=6,
        bg=LINE,
        highlightthickness=0
    )

    meter.create_rectangle(
        0,
        0,
        130 * max(0, min(score, 1)),
        6,
        fill=ACCENT,
        width=0
    )

    meter.pack(pady=(4, 0))


    def paint(c):

        for w in (
            row,
            info,
            right,
            *info.winfo_children(),
            *right.winfo_children()
        ):

            w.configure(bg=c)


    def pick(_):

        var.set(title)
        run()


    for w in (
        row,
        *row.winfo_children(),
        *info.winfo_children()
    ):

        w.bind(
            "<Enter>",
            lambda e: paint(HOVER)
        )

        w.bind(
            "<Leave>",
            lambda e: paint(CARD)
        )

        w.bind(
            "<Button-1>",
            pick
        )


# ---------- Clear results ----------
def clear():

    for w in inner.winfo_children():
        w.destroy()

    canvas.yview_moveto(0)


# ---------- Reset ----------
def reset():

    var.set("")

    combo["values"] = all_titles

    clear()

    kicker.config(text="WELCOME")

    h_title.config(
        text="What should you watch next?"
    )

    h_meta.config(
        text=f"{len(all_titles):,} films indexed"
    )

    h_plot.config(
        text="Choose a film on the left to see its ten closest matches."
    )


# ---------- Find recommendations ----------
def run(_=None):

    t = var.get().strip()

    if t not in index_of:

        m = [
            x for x in all_titles
            if t.lower() in x.lower()
        ] if t else []

        if len(m) != 1:

            messagebox.showwarning(
                "Select a film",
                "Choose a film from the list."
            )

            return

        t = m[0]

        var.set(t)


    clear()

    i = index_of[t]

    kicker.config(
        text="BECAUSE YOU LIKED"
    )

    h_title.config(
        text=t
    )

    h_meta.config(
        text=meta(i)
    )

    p = plot(i)

    h_plot.config(
        text=(p[:260] + "…") if len(p) > 260 else p
    )


    for r, (j, s) in enumerate(
        recommend(t),
        1
    ):

        make_row(r, j, s)


    existing = history.get(0, "end")

    if t in existing:
        history.delete(existing.index(t))

    history.insert(0, t)


# ---------- History selection ----------
def from_history(_):

    sel = history.curselection()

    if sel:

        var.set(history.get(sel[0]))

        run()


# ---------- Search filtering ----------
def filter_list(e):

    if e.keysym in (
        "Return",
        "Up",
        "Down",
        "Escape",
        "Tab"
    ):
        return

    q = var.get().lower()

    combo["values"] = (
        [x for x in all_titles if q in x.lower()][:200]
        if q else all_titles
    )


history.bind(
    "<<ListboxSelect>>",
    from_history
)

combo.bind(
    "<KeyRelease>",
    filter_list
)

combo.bind(
    "<Return>",
    run
)

combo.bind(
    "<<ComboboxSelected>>",
    run
)


# ---------- Start application ----------
combo.focus_set()

reset()

root.mainloop()
